# 08 — Check model inputs and assumptions

This notebook shows how the two model datasets are built and checks them before estimation. It does not fit the final models and it does not create figures. Notebook 09 repeats the preparation explicitly so it can be read on its own.


## 1. Paths and analysis period


In [ ]:
from pathlib import Path
import json
import platform
from datetime import datetime, timezone

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "ANAL":
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "ANAL").is_dir():
    raise RuntimeError("Start the notebook from the repository root or the ANAL directory.")

DATA_DIR = PROJECT_DIR / "ANAL" / "data"
FEATURE_DIR = DATA_DIR / "routing" / "features"
PANEL_PATH = DATA_DIR / "raster_quarter_panel_100m.parquet"
FIRMS_PATH = DATA_DIR / "firms_assigned_100m.geoparquet"
BIRTHS_BY_GROUP_PATH = DATA_DIR / "births_by_fachgruppe_100m.parquet"
RESULT_DIR = DATA_DIR / "models"
CACHE_DIR = DATA_DIR / "cache"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2016
END_YEAR = 2025
LAG_YEAR = 2015

print(f"Project: {PROJECT_DIR}")
print(f"Analysis period: {START_YEAR}–{END_YEAR}")
print(f"Model results: {RESULT_DIR}")


In [ ]:
input_files = pd.DataFrame([
    {"input": "cell-quarter panel", "path": PANEL_PATH, "exists": PANEL_PATH.exists()},
    {"input": "firm records", "path": FIRMS_PATH, "exists": FIRMS_PATH.exists()},
    {"input": "births by Fachgruppe", "path": BIRTHS_BY_GROUP_PATH, "exists": BIRTHS_BY_GROUP_PATH.exists()},
])
display(input_files)
if not input_files["exists"].all():
    raise FileNotFoundError("At least one main model input is missing.")


## 2. Check required files and columns


In [ ]:
def require_columns(path, required):
    if not path.exists():
        raise FileNotFoundError(path)
    available = set(pq.ParquetFile(path).schema_arrow.names)
    missing = sorted(set(required) - available)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

require_columns(PANEL_PATH, ["grid_id", "municipality_id", "year", "quarter", "period", "births", "active_firms_tminus1"])
require_columns(FIRMS_PATH, ["firm_id", "founding_date", "exit_date", "exit_observed", "Fachgruppe_ID", "Sparte_ID", "Sparte_Text"])

for year in range(LAG_YEAR, END_YEAR + 1):
    year_dir = FEATURE_DIR / str(year)
    require_columns(year_dir / "accessibility_potentials_100m.parquet", [
        "grid_id", "year", "quarter", "period", "own_cell_pop", "own_cell_firms",
        "pop_access_15min", "existing_firms_access_15min",
    ])
    require_columns(year_dir / "pedestrian_accessibility_quarter_100m.parquet", [
        "grid_id", "year", "quarter", "period", "own_cell_walk_pop", "own_cell_walk_firms",
        "walk_pop_10min", "walk_firms_10min", "walk_pt_routes_10min", "pt_ohne_haltestelle",
    ])
    require_columns(year_dir / "nearest_infrastructure_100m.parquet", ["grid_id", "year", "tt_motorway_exit_min"])

for year in range(START_YEAR, END_YEAR + 1):
    require_columns(FEATURE_DIR / str(year) / "firm_accessibility_quarter_100m.parquet", [
        "firm_id", "grid_id_100m", "Fachgruppe_ID", "year", "quarter", "period",
        "included_in_lagged_stock", "own_cell_pop", "own_cell_firms",
        "own_cell_same_fachgruppe_firms", "pop_access_15min",
        "existing_firms_access_15min", "same_fachgruppe_firms_access_15min",
    ])

print("Input check passed for all required years and columns.")


## 3. Build the founding-model data


### 3.1 Load and join the inputs


In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

panel_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{PANEL_PATH.as_posix()}') WHERE year BETWEEN {LAG_YEAR} AND {END_YEAR}"
).fetchone()[0]

founding = con.execute(f"""
SELECT p.grid_id, p.municipality_id, p.year, p.quarter, p.period,
       p.births, p.active_firms_tminus1,
       a.own_cell_pop, a.own_cell_firms,
       a.pop_access_15min, a.existing_firms_access_15min,
       w.own_cell_walk_pop, w.own_cell_walk_firms,
       w.walk_pop_10min, w.walk_firms_10min,
       w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
       n.tt_motorway_exit_min
FROM read_parquet('{PANEL_PATH.as_posix()}') p
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/'accessibility_potentials_100m.parquet').as_posix()}') a
  USING (grid_id, year, quarter, period)
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/'pedestrian_accessibility_quarter_100m.parquet').as_posix()}') w
  USING (grid_id, year, quarter, period)
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/'nearest_infrastructure_100m.parquet').as_posix()}') n
  USING (grid_id, year)
WHERE p.year BETWEEN {LAG_YEAR} AND {END_YEAR}
ORDER BY p.grid_id, p.year, p.quarter
""").df()

if len(founding) != panel_rows:
    raise ValueError("The founding-data joins lost or duplicated panel rows.")
if founding.duplicated(["grid_id", "year", "quarter"]).any():
    raise ValueError("Duplicate cell-quarter rows after the joins.")

print(f"Panel input: {panel_rows:,} rows")
print(f"Joined founding data: {founding.shape[0]:,} rows × {founding.shape[1]} columns")
print(f"Cells: {founding['grid_id'].nunique():,}; quarters: {founding['period'].nunique()}")


### 3.2 Remove incomplete routing rows


In [ ]:
founding_route_columns = [
    "pop_access_15min", "existing_firms_access_15min", "walk_pop_10min", "walk_firms_10min",
    "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min",
]
incomplete = founding[founding_route_columns].isna().any(axis=1)
print(f"Rows with incomplete routing: {incomplete.sum():,} ({incomplete.mean():.3%})")
founding = founding.loc[~incomplete].copy()
print(f"Rows retained: {len(founding):,}")


### 3.3 Create neighbourhood variables


In [ ]:
# Non-overlapping neighbourhoods. The origin cell is removed.
founding["car_pop_ring_0_15"] = (founding["pop_access_15min"] - founding["own_cell_pop"]).clip(lower=0)
founding["car_firms_ring_0_15"] = (founding["existing_firms_access_15min"] - founding["own_cell_firms"]).clip(lower=0)
founding["walk_pop_ring_0_10"] = (founding["walk_pop_10min"] - founding["own_cell_walk_pop"]).clip(lower=0)
founding["walk_firms_ring_0_10"] = (founding["walk_firms_10min"] - founding["own_cell_walk_firms"]).clip(lower=0)

founding["log_pop_access_ring_0_15"] = np.log1p(founding["car_pop_ring_0_15"])
founding["log_firms_relative_car_ring_0_15"] = (
    np.log1p(founding["car_firms_ring_0_15"]) - founding["log_pop_access_ring_0_15"]
)
founding["log_walk_pop_ring_0_10"] = np.log1p(founding["walk_pop_ring_0_10"])
founding["log_firms_relative_walk_ring_0_10"] = (
    np.log1p(founding["walk_firms_ring_0_10"]) - founding["log_walk_pop_ring_0_10"]
)

print("Created car and walking population-mass and relative-firm-density variables.")


### 3.4 Create population growth and lags


In [ ]:
founding = founding.sort_values(["grid_id", "year", "quarter"])
period_number = founding["year"] * 4 + founding["quarter"]
previous_period = period_number.groupby(founding["grid_id"], sort=False).shift(4)
previous_population = founding.groupby("grid_id", sort=False)["own_cell_pop"].shift(4)
previous_population = previous_population.where(period_number - previous_period == 4)

founding["log_own_firms"] = np.log1p(founding["active_firms_tminus1"])
founding["log_own_pop"] = np.log1p(previous_population)
founding["population_growth_yoy"] = np.log1p(founding["own_cell_pop"]) - np.log1p(previous_population)
founding["log_tt_motorway_exit"] = np.log1p(founding["tt_motorway_exit_min"])

before = len(founding)
founding = founding.loc[founding["year"] >= START_YEAR].copy()
print(f"Removed {before-len(founding):,} lead-in rows from {LAG_YEAR}.")


### 3.5 Create the design matrix


In [ ]:
founding_terms = [
    "log_own_firms", "log_own_pop",
    "log_pop_access_ring_0_15", "log_firms_relative_car_ring_0_15",
    "log_walk_pop_ring_0_10", "log_firms_relative_walk_ring_0_10",
    "population_growth_yoy", "log_tt_motorway_exit",
    "walk_pt_routes_10min", "pt_ohne_haltestelle",
]

before = len(founding)
founding = founding.dropna(subset=founding_terms).copy()
period_dummies = pd.get_dummies(founding["period"], prefix="period", drop_first=True, dtype="float64")
X_founding = pd.concat([founding[founding_terms].astype("float64"), period_dummies], axis=1)
X_founding.insert(0, "const", 1.0)
y_founding = founding["births"].astype("float64")
founding_clusters = founding["grid_id"]

if not np.isfinite(X_founding.to_numpy()).all(): raise ValueError("Non-finite founding regressors.")
if (y_founding < 0).any() or not np.allclose(y_founding, np.floor(y_founding)):
    raise ValueError("Births must be non-negative integer counts.")

print(f"Rows removed for missing model values: {before-len(founding):,}")
print(f"Final founding sample: {len(founding):,} rows × {X_founding.shape[1]} design columns")
print(f"Births: {int(y_founding.sum()):,}; zero-count rows: {(y_founding==0).mean():.2%}")
print(f"Cell clusters: {founding_clusters.nunique():,}")


### 3.6 Founding-model checks


In [ ]:
def correlation_report(frame, terms, threshold=0.70):
    corr = frame[terms].corr()
    pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
    return pairs[pairs.abs() >= threshold].sort_values(key=abs, ascending=False).rename("correlation")

def condition_number(frame, terms):
    eigenvalues = np.linalg.eigvalsh(frame[terms].corr().to_numpy())
    return np.inf if eigenvalues[0] <= 0 else float(np.sqrt(eigenvalues[-1] / eigenvalues[0]))


In [ ]:
founding_correlations = correlation_report(founding, founding_terms)
founding_summary = pd.Series({
    "rows": len(founding), "columns_in_design_matrix": X_founding.shape[1],
    "cells": founding["grid_id"].nunique(), "quarters": founding["period"].nunique(),
    "births": int(y_founding.sum()), "zero_share": float((y_founding==0).mean()),
    "birth_mean": float(y_founding.mean()), "birth_variance": float(y_founding.var()),
    "variance_to_mean": float(y_founding.var()/y_founding.mean()),
    "condition_number": condition_number(founding, founding_terms),
})
display(founding_summary.to_frame("value"))
print("Correlations with absolute value at least 0.70:")
print(founding_correlations.to_string() if len(founding_correlations) else "None")
founding_summary.to_csv(RESULT_DIR/"founding_input_diagnostics.csv", header=["value"])
founding_correlations.to_csv(RESULT_DIR/"founding_high_correlations.csv", header=True)
print("Saved 2 diagnostic files.")


## 4. Build the survival-model data


### 4.1 Load and join firm-quarter inputs


In [ ]:
firm_feature_glob = (FEATURE_DIR / "*" / "firm_accessibility_quarter_100m.parquet").as_posix()
raw_survival_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{firm_feature_glob}') WHERE year BETWEEN {START_YEAR} AND {END_YEAR}"
).fetchone()[0]

spells = con.execute(f"""
SELECT a.firm_id AS standort_id, a.grid_id_100m AS grid_id,
       CAST(a.Fachgruppe_ID AS VARCHAR) AS Fachgruppe_ID,
       a.year, a.quarter, a.period, a.included_in_lagged_stock,
       a.own_cell_pop, a.own_cell_firms, a.own_cell_same_fachgruppe_firms,
       a.pop_access_15min, a.existing_firms_access_15min,
       a.same_fachgruppe_firms_access_15min,
       w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
       n.tt_motorway_exit_min
FROM read_parquet('{firm_feature_glob}') a
JOIN read_parquet('{(FEATURE_DIR/'*'/'pedestrian_accessibility_quarter_100m.parquet').as_posix()}') w
  ON a.grid_id_100m=w.grid_id AND a.year=w.year AND a.quarter=w.quarter
JOIN read_parquet('{(FEATURE_DIR/'*'/'nearest_infrastructure_100m.parquet').as_posix()}') n
  ON a.grid_id_100m=n.grid_id AND a.year=n.year
WHERE a.year BETWEEN {START_YEAR} AND {END_YEAR}
ORDER BY a.firm_id, a.year, a.quarter
""").df()

if len(spells) != raw_survival_rows: raise ValueError("The survival joins lost or duplicated rows.")
if spells.duplicated(["standort_id", "year", "quarter"]).any(): raise ValueError("Duplicate firm-quarter rows.")
print(f"Firm-quarter input: {raw_survival_rows:,} rows")
print(f"Joined survival data: {spells.shape[0]:,} rows × {spells.shape[1]} columns")
print(f"Locations: {spells['standort_id'].nunique():,}")


### 4.2 Remove incomplete firm histories


In [ ]:
survival_route_columns = [
    "pop_access_15min", "existing_firms_access_15min", "same_fachgruppe_firms_access_15min",
    "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min",
]
bad_ids = spells.loc[spells[survival_route_columns].isna().any(axis=1), "standort_id"].unique()
missing_group_ids = spells.loc[spells["Fachgruppe_ID"].isna(), "standort_id"].unique()
excluded_ids = np.union1d(bad_ids, missing_group_ids)
spells = spells.loc[~spells["standort_id"].isin(excluded_ids)].copy()
print(f"Locations removed with incomplete histories: {len(excluded_ids):,}")
print(f"Rows retained: {len(spells):,}; locations retained: {spells['standort_id'].nunique():,}")


### 4.3 Create survival intervals


In [ ]:
firm_dates = pd.read_parquet(FIRMS_PATH, columns=[
    "firm_id", "founding_date", "exit_date", "exit_observed", "Sparte_ID", "Sparte_Text",
]).rename(columns={"firm_id":"standort_id", "Sparte_ID":"sparte", "Sparte_Text":"sparte_name"})
if firm_dates["standort_id"].duplicated().any(): raise ValueError("firm_id is not unique.")
firm_dates["founding_date"] = pd.to_datetime(firm_dates["founding_date"], errors="coerce")
firm_dates["exit_date"] = pd.to_datetime(firm_dates["exit_date"], errors="coerce")
firm_dates["sparte"] = firm_dates["sparte"].astype("string")
spells = spells.merge(firm_dates, on="standort_id", how="left", validate="many_to_one")
if spells[["founding_date", "sparte", "sparte_name"]].isna().any().any():
    raise ValueError("A routed location lacks dates or an official sector mapping.")

quarter_index = pd.PeriodIndex(spells["period"], freq="Q").asi8
founding_index = spells["founding_date"].dt.to_period("Q").array.asi8
spells["start"] = quarter_index - founding_index
spells["stop"] = spells["start"] + 1
exit_quarter = spells["exit_date"].dt.to_period("Q").astype("string")
spells["event"] = (spells["exit_observed"].fillna(False) & spells["period"].eq(exit_quarter)).astype("int8")
spells = spells.sort_values(["standort_id", "year", "quarter"])

if (spells["start"] < 0).any(): raise ValueError("A survival interval starts before firm founding.")
if (spells.groupby("standort_id")["event"].sum() > 1).any(): raise ValueError("A location has multiple exits.")
sequence = spells["year"]*4 + spells["quarter"]
if sequence.groupby(spells["standort_id"]).diff().dropna().ne(1).any(): raise ValueError("Firm histories contain gaps.")

entry_age = spells.groupby("standort_id")["start"].min()
print(f"Observed exits: {int(spells['event'].sum()):,}")
print(f"Left-truncated locations: {(entry_age>0).sum():,} ({(entry_age>0).mean():.2%})")


### 4.4 Create Cox covariates


In [ ]:
focal = spells["included_in_lagged_stock"].astype("float64")
spells["own_firms"] = (spells["own_cell_firms"] - focal).clip(lower=0)
spells["own_same"] = (spells["own_cell_same_fachgruppe_firms"] - focal).clip(lower=0)
spells["own_other"] = (spells["own_firms"] - spells["own_same"]).clip(lower=0)

all_firms = (spells["existing_firms_access_15min"] - focal).clip(lower=0)
same_firms = (spells["same_fachgruppe_firms_access_15min"] - focal).clip(lower=0)
other_firms = (all_firms - same_firms).clip(lower=0)
pop_ring = (spells["pop_access_15min"] - spells["own_cell_pop"]).clip(lower=0)
same_ring = (same_firms - spells["own_same"]).clip(lower=0)
other_ring = (other_firms - spells["own_other"]).clip(lower=0)

spells["log_own_pop"] = np.log1p(spells["own_cell_pop"])
spells["log_own_same"] = np.log1p(spells["own_same"])
spells["log_own_other"] = np.log1p(spells["own_other"])
spells["log_pop_ring_0_15"] = np.log1p(pop_ring)
spells["log_same_relative_ring_0_15"] = np.log1p(same_ring) - spells["log_pop_ring_0_15"]
spells["log_other_relative_ring_0_15"] = np.log1p(other_ring) - spells["log_pop_ring_0_15"]
spells["log_tt_motorway_exit"] = np.log1p(spells["tt_motorway_exit_min"])
spells["calendar_year"] = spells["year"] - START_YEAR
print("Removed the focal location and created the Cox-model covariates.")


### 4.5 Create the Cox model frame


In [ ]:
survival_terms = [
    "log_own_pop", "log_own_same", "log_own_other", "log_pop_ring_0_15",
    "log_same_relative_ring_0_15", "log_other_relative_ring_0_15",
    "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle", "calendar_year",
]
survival_frame = spells[[
    "standort_id", "grid_id", "Fachgruppe_ID", "sparte", "sparte_name", "year", "period",
    "start", "stop", "event", *survival_terms,
]].copy()
if not np.isfinite(survival_frame[survival_terms].to_numpy()).all(): raise ValueError("Non-finite Cox covariates.")
print(f"Final survival sample: {survival_frame.shape[0]:,} intervals × {survival_frame.shape[1]} columns")
print(f"Locations: {survival_frame['standort_id'].nunique():,}; exits: {int(survival_frame['event'].sum()):,}")
print(f"Cells: {survival_frame['grid_id'].nunique():,}; Fachgruppe strata: {survival_frame['Fachgruppe_ID'].nunique()}")


### 4.6 Survival-model checks


In [ ]:
survival_correlations = correlation_report(survival_frame, survival_terms)
events_by_group = survival_frame.groupby("Fachgruppe_ID")["event"].agg(["sum", "size"])
survival_summary = pd.Series({
    "intervals": len(survival_frame), "columns": survival_frame.shape[1],
    "locations": survival_frame["standort_id"].nunique(), "events": int(survival_frame["event"].sum()),
    "cells": survival_frame["grid_id"].nunique(), "fachgruppe_strata": len(events_by_group),
    "strata_without_events": int((events_by_group["sum"]==0).sum()),
    "strata_below_30_events": int((events_by_group["sum"]<30).sum()),
    "events_per_covariate": float(survival_frame["event"].sum()/len(survival_terms)),
    "condition_number": condition_number(survival_frame, survival_terms),
})
display(survival_summary.to_frame("value"))
print("Correlations with absolute value at least 0.70:")
print(survival_correlations.to_string() if len(survival_correlations) else "None")
survival_summary.to_csv(RESULT_DIR/"survival_input_diagnostics.csv", header=["value"])
survival_correlations.to_csv(RESULT_DIR/"survival_high_correlations.csv", header=True)
print("Saved 2 diagnostic files.")


## 5. What remains to be checked after fitting

The count variance is only a first indication of overdispersion. Notebook 09 compares Poisson and NB2 with a boundary likelihood-ratio test. The Cox proportional-hazards assumption also requires a fitted model; notebook 09 saves a Schoenfeld-residual screen for notebook 10.
